# 01 — Data Collection & Audit

**Purpose:** Pull, register, and validate all external datasets used across the three analytical modules. Every source is documented; every file is written to `data/raw/`.

---

## Sources targeted
| # | Dataset | Module(s) | Download method |
|---|---|---|---|
| 1 | Lighting Global PAYGo MAI | 1 | Manual download → `data/raw/lighting_global_mai/` |
| 2 | World Bank Global Findex 2021 | 1, 2 | World Bank API |
| 3 | GSMA Mobile Money KPIs | 1 | Manual download |
| 4 | DHS country files (KE, UG, TZ, ET) | 1, 2 | Manual download after registration |
| 5 | Meta Relative Wealth Index | 2 | Humanitarian Data Exchange API |
| 6 | FinAccess Kenya 2021 | 2 | Manual download |
| 7 | GOGLA PERFORM benchmarks | 2, 3 | PDF extraction → structured CSV |

In [7]:
import os
import requests
import pandas as pd
import json
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
ROOT = Path("C:/Users/eric.njiraini/Documents/My Projects at a Go/paygo-cookstove-market-entry")
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
SIM  = ROOT / "data" / "simulated"

for p in [RAW, PROC, SIM]:
    p.mkdir(parents=True, exist_ok=True)

COUNTRIES = {
    "KE": "Kenya",
    "UG": "Uganda",
    "TZ": "Tanzania",
    "ET": "Ethiopia"
}

print("Directories ready.")

Directories ready.


## 1.1 World Bank Findex 2021 — Financial Inclusion Indicators

Pulling key financial inclusion indicators via the World Bank Data API for all four countries.

In [8]:
# World Bank API — indicator codes
# FX.OWN.TOTL.ZS  = Account ownership at a financial institution (% age 15+)
# FB.CBK.BRCH.P5  = Bank branches per 100,000 adults
# IT.CEL.SETS.P2  = Mobile cellular subscriptions per 100 people

WB_INDICATORS = {
    "FX.OWN.TOTL.ZS": "financial_account_pct",
    "FB.CBK.BRCH.P5":  "bank_branches_per_100k",
    "IT.CEL.SETS.P2":  "mobile_subs_per_100",
    "EG.CFT.ACCS.ZS":  "clean_fuel_access_pct",   # Clean fuels for cooking
    "SP.URB.TOTL.IN.ZS": "urban_pop_pct",
}

WB_BASE = "https://api.worldbank.org/v2/country/{iso}/indicator/{ind}?format=json&mrv=1"

records = []

for iso, name in COUNTRIES.items():
    for ind_code, ind_label in WB_INDICATORS.items():
        url = WB_BASE.format(iso=iso, ind=ind_code)
        resp = requests.get(url, timeout=60)
        data = resp.json()
        if len(data) > 1 and data[1]:
            val = data[1][0].get("value")
            year = data[1][0].get("date")
            records.append({"iso": iso, "country": name, "indicator": ind_label, "value": val, "year": year})
        else:
            print(f"  No data: {iso} / {ind_code}")

wb_long = pd.DataFrame(records)
wb_wide = wb_long.pivot_table(index=["iso","country"], columns="indicator", values="value").reset_index()

wb_wide.to_csv(RAW / "wb_findex_indicators.csv", index=False)
print("World Bank indicators saved.")
wb_wide

World Bank indicators saved.


indicator,iso,country,bank_branches_per_100k,clean_fuel_access_pct,financial_account_pct,mobile_subs_per_100,urban_pop_pct
0,ET,Ethiopia,14.489596,7.1,48.830277,65.082509,23.622284
1,KE,Kenya,4.480817,31.6,90.119917,126.477706,31.893783
2,TZ,Tanzania,2.347735,9.8,59.827525,126.559163,36.168446
3,UG,Uganda,2.259381,1.8,72.827376,83.165031,31.277770


## 1.2 Meta Relative Wealth Index — Humanitarian Data Exchange

The RWI provides sub-national wealth estimates at ~2.4km grid resolution, derived from Facebook connectivity data. We pull the country-level files for all four countries.

In [9]:
# HDX dataset slugs for Meta RWI
# Full files are large — download manually from:
# https://data.humdata.org/dataset/relative-wealth-index
# Place CSV files in data/raw/rwi/

RWI_DIR = RAW / "rwi"
RWI_DIR.mkdir(exist_ok=True)

RWI_FILES = {
    "KE": "ken_relative_wealth_index.csv",
    "UG": "uga_relative_wealth_index.csv",
    "TZ": "tza_relative_wealth_index.csv",
    "ET": "eth_relative_wealth_index.csv",
}

rwi_frames = {}
for iso, fname in RWI_FILES.items():
    fpath = RWI_DIR / fname
    if fpath.exists():
        df = pd.read_csv(fpath)
        rwi_frames[iso] = df
        print(f"  {iso}: {len(df):,} grid cells loaded")
    else:
        print(f"  ⚠ {iso}: file not found — download from HDX and place in {RWI_DIR}")

print("\nRWI check complete.")

  KE: 29,483 grid cells loaded
  UG: 25,404 grid cells loaded
  TZ: 63,689 grid cells loaded
  ET: 85,142 grid cells loaded

RWI check complete.


## 1.3 GOGLA Benchmark Extraction

GOGLA's PAYGo PERFORM reports publish sector-level repayment and default benchmarks. Since these are published as PDFs, we encode the key figures here as structured constants used in simulation (Module 2 & 3).

In [10]:
# ── GOGLA PAYGo PERFORM benchmark constants ───────────────────────────────────
# Source: GOGLA PAYGo PERFORM reports (2022–2023 editions)
# These are used to calibrate the synthetic customer ledger in Module 2 & 3.

GOGLA_BENCHMARKS = {
    "sector_avg_repayment_rate": 0.72,        # 72% of financed amount recovered
    "sector_default_rate": 0.28,              # Implied from above
    "best_quartile_repayment": 0.90,
    "worst_quartile_repayment": 0.58,
    "downpayment_threshold_high": 0.10,       # >10% DP → better repayment
    "downpayment_threshold_low": 0.05,        # <5% DP → worst repayment cohort
    "typical_contract_months_range": (12, 36),
    "lockout_trigger_days": 3,                # Consecutive missed days → lockout
}

import json
(RAW / "gogla_benchmarks.json").write_text(json.dumps(GOGLA_BENCHMARKS, indent=2))

pd.DataFrame([GOGLA_BENCHMARKS]).T.rename(columns={0: "value"})

,value
sector_avg_repayment_rate,0.72
sector_default_rate,0.28
best_quartile_repayment,0.9
worst_quartile_repayment,0.58
downpayment_threshold_high,0.1
downpayment_threshold_low,0.05
typical_contract_months_range,"(12, 36)"
lockout_trigger_days,3


## 1.4 Data Audit

Summary of what's available locally vs. still needed.

In [11]:
MANUAL_DOWNLOADS = [
    {"dataset": "Lighting Global PAYGo MAI",     "path": "data/raw/lighting_global_mai/", "url": "https://www.lightingglobal.org/work-with-us/paygo-mai"},
    {"dataset": "GSMA Mobile Money KPIs",         "path": "data/raw/gsma/",               "url": "https://www.gsma.com/solutions-and-impact/connectivity-programmes/mobile-for-development/fintech/mobile-money"},
    {"dataset": "DHS Kenya (KEHR8AFL)",           "path": "data/raw/dhs/KE/",             "url": "https://dhsprogram.com/data/"},
    {"dataset": "DHS Uganda (UGHR7BFL)",          "path": "data/raw/dhs/UG/",             "url": "https://dhsprogram.com/data/"},
    {"dataset": "DHS Tanzania (TZHR7AFL)",        "path": "data/raw/dhs/TZ/",             "url": "https://dhsprogram.com/data/"},
    {"dataset": "DHS Ethiopia (ETHR8AFL)",        "path": "data/raw/dhs/ET/",             "url": "https://dhsprogram.com/data/"},
    {"dataset": "Meta RWI — all 4 countries",     "path": "data/raw/rwi/",               "url": "https://data.humdata.org/dataset/relative-wealth-index"},
    {"dataset": "FinAccess Kenya 2021",           "path": "data/raw/finaccess/",         "url": "https://www.knbs.or.ke"},
]

audit = pd.DataFrame(MANUAL_DOWNLOADS)
audit["exists"] = audit["path"].apply(lambda p: (ROOT / p).exists())
print(audit.to_string(index=False))

                   dataset                          path                                                                                                           url  exists
 Lighting Global PAYGo MAI data/raw/lighting_global_mai/                                                         https://www.lightingglobal.org/work-with-us/paygo-mai   False
    GSMA Mobile Money KPIs                data/raw/gsma/ https://www.gsma.com/solutions-and-impact/connectivity-programmes/mobile-for-development/fintech/mobile-money   False
      DHS Kenya (KEHR8AFL)              data/raw/dhs/KE/                                                                                  https://dhsprogram.com/data/   False
     DHS Uganda (UGHR7BFL)              data/raw/dhs/UG/                                                                                  https://dhsprogram.com/data/   False
   DHS Tanzania (TZHR7AFL)              data/raw/dhs/TZ/                                                                     

---

**Next:** `02_country_comparison.ipynb` — build the country-level comparison across all dimensions.